# KrushiAI: SMART CROP RECOMMENDATION SYSTEM

## 1. Introduction

This project aims to build a smart crop recommendation system, KrushiAI, which suggests the best crop to grow based on soil and environmental conditions. By leveraging machine learning, this system can help farmers make informed decisions, leading to better yields and more sustainable agricultural practices.

## 2. Importing Libraries

First, let's import the necessary libraries for data analysis, visualization, and machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.neighbors import KNeighborsClassifier
import pickle
import warnings
warnings.filterwarnings('ignore')

## 3. Data Loading and Initial Analysis

We will load the dataset and perform an initial analysis to understand its structure and content.

In [ ]:
df = pd.read_csv('Crop_recommendation.csv')

In [ ]:
df.head()

The dataset consists of 2200 rows and 8 columns. Each row represents a set of conditions, and the columns are:
- **N, P, K**: The amount of Nitrogen, Phosphorous, and Potassium in the soil.
- **temperature, humidity, rainfall**: Environmental conditions.
- **ph**: The pH level of the soil.
- **label**: The type of crop that is best suited for the given conditions. This is our target variable.

In [ ]:
df.info()

## 4. Data Quality Check

Let's check for any missing values or duplicate rows in our dataset.

In [ ]:
print('Missing values in each column:')
print(df.isnull().sum())

In [ ]:
print('\nNumber of duplicate rows:', df.duplicated().sum())

The dataset is clean, with no missing values or duplicate rows.

## 5. Exploratory Data Analysis (EDA)

Now, let's explore the data to find patterns and relationships between the features.

In [ ]:
df['label'].value_counts()

The dataset is well-balanced, with 100 samples for each of the 22 crops.

In [ ]:
plt.figure(figsize=[10,5],dpi = 100)
sns.heatmap(df.select_dtypes(include='number').corr(), annot=True, cmap='viridis')
plt.title('Correlation Matrix')
plt.show()

The heatmap shows the correlation between the features. We can see some correlations, such as between K and P, and humidity and rainfall.

### Visualizing Feature Distributions

Let's visualize the distribution of each feature for different crops to understand their requirements.

In [ ]:
def plot_feature_distribution(feature):
    plt.figure(figsize=(15, 7))
    sns.boxplot(x='label', y=feature, data=df)
    plt.xticks(rotation=90)
    plt.title(f'Distribution of {feature} for each Crop')
    plt.show()

features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
for feature in features:
    plot_feature_distribution(feature)

These boxplots show the range of conditions required for each crop. For example, we can see that cotton requires a high amount of N, while rice requires a high amount of rainfall.

## 6. Data Preprocessing

Before training our models, we need to preprocess the data. This includes separating features and the target variable, encoding the categorical target variable, and scaling the features.

In [ ]:
# Separate features and target
X = df.drop('label', axis=1)
y = df['label']

# Encode the target variable
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 7. Model Training and Evaluation

We will now train several machine learning models and evaluate their performance.

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(criterion="entropy", random_state=2, max_depth=5),
    'Naive Bayes': GaussianNB(),
    'SVM': SVC(gamma='auto'),
    'Logistic Regression': LogisticRegression(random_state=2),
    'Random Forest': RandomForestClassifier(n_estimators=20, random_state=5),
    'XGBoost': xgb.XGBClassifier(),
    'KNN': KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
}

acc = []
model_names = []

for name, model in models.items():
    if name in ['SVM', 'Logistic Regression', 'KNN']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    acc.append(accuracy)
    model_names.append(name)
    
    print(f"{name}'s Accuracy is: {accuracy*100:.2f}%")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

### Accuracy Comparison

In [ ]:
plt.figure(figsize=[10,5],dpi = 100)
plt.title('Accuracy Comparison')
plt.xlabel('Algorithm')
plt.ylabel('Accuracy')
sns.barplot(x=model_names, y=acc, palette='dark')
plt.xticks(rotation=45)
for i, accuracy in enumerate(acc):
    plt.text(i, accuracy + 0.01, f'{accuracy:.2%}', ha='center')
plt.show()

### Per-Crop Accuracy for the Best Model (Random Forest)

In [ ]:
def plot_per_crop_accuracy(model, model_name):
    if model_name in ['SVM', 'Logistic Regression', 'KNN']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
    accuracy_per_crop = []
    crop_labels = le.inverse_transform(sorted(np.unique(y_test)))
    for crop_label in crop_labels:
        indices = (y_test == le.transform([crop_label])[0])
        accuracy = accuracy_score(y_test[indices], y_pred[indices])
        accuracy_per_crop.append(accuracy)

    plt.figure(figsize=(10, 6))
    sns.barplot(x=crop_labels, y=accuracy_per_crop, palette='viridis')
    plt.xticks(rotation=90)
    plt.title(f'Accuracy of {model_name} Model for Each Crop')
    plt.ylabel('Accuracy')
    plt.xlabel('Crop')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_per_crop_accuracy(models['Random Forest'], 'Random Forest')

## 8. Model Saving

Now we will save the trained models for future use. We will save the best performing model, Random Forest, as well as the Label Encoder and Scaler.

In [ ]:
# Save the Random Forest model
with open('RandomForest.pkl', 'wb') as f:
    pickle.dump(models['Random Forest'], f)

# Save the Label Encoder
with open('LabelEncoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Save the Scaler
with open('Scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

## 9. Making a Prediction

Let's use our trained Random Forest model to make a prediction on new data.

In [ ]:
data = np.array([[104, 18, 30, 23.603016, 60.3, 6.7, 140.91]])
prediction_encoded = models['Random Forest'].predict(data)
prediction = le.inverse_transform(prediction_encoded)
print(f"The recommended crop is: {prediction[0]}")

In [ ]:
data = np.array([[83, 45, 60, 28, 70.3, 7.0, 150.9]])
prediction_encoded = models['Random Forest'].predict(data)
prediction = le.inverse_transform(prediction_encoded)
print(f"The recommended crop is: {prediction[0]}")

## 10. Conclusion

In this project, we successfully built a crop recommendation system using various machine learning models. The Random Forest model performed the best with an accuracy of 99.77%. We also performed exploratory data analysis to understand the data and saved the best model for future use. This system can be a valuable tool for farmers to make data-driven decisions.